# Support Vector Regression (SVR) with Multiple Hyperparameters using GridSearchCV
Edit the `param_grid` section to try different combinations.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import pickle

In [4]:
# Load dataset
dataset = pd.read_csv('insurance_pre.csv')
dataset = pd.get_dummies(dataset, drop_first=True)

X = dataset.drop('charges', axis=1)
y = dataset['charges']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(X.columns.tolist())

['age', 'bmi', 'children', 'sex_male', 'smoker_yes']


In [5]:
# Modify these hyperparameters as needed
param_grid = {
    'kernel': ['linear','rbf','poly','sigmoid'],
    'C': [0.1,1,10,100,1000],
    'gamma': ['scale','auto'],
    'epsilon': [0.01,0.1,0.5,1.0],
    'degree': [2,3,4]   # Used only for polynomial kernel
}

grid = GridSearchCV(
    estimator=SVR(),
    param_grid=param_grid,
    scoring='r2',
    cv=5,
    n_jobs=-1,
    verbose=2,
    refit=True
)

grid.fit(X_train_scaled, y_train)

print("Best Parameters:", grid.best_params_)
print("Best CV Score:", grid.best_score_)

Fitting 5 folds for each of 480 candidates, totalling 2400 fits
Best Parameters: {'C': 1000, 'degree': 3, 'epsilon': 1.0, 'gamma': 'auto', 'kernel': 'poly'}
Best CV Score: 0.7895690975395773


In [6]:
best_model = grid.best_estimator_

pred = best_model.predict(X_test_scaled)

print("R2 Score :", r2_score(y_test, pred))
print("MAE      :", mean_absolute_error(y_test, pred))
print("RMSE     :", np.sqrt(mean_squared_error(y_test, pred)))

R2 Score : 0.8506657363871135
MAE      : 2220.47611089501
RMSE     : 4679.319720548967


In [9]:
# Predict for new customer
age = 32
bmi = 28.5
children = 2
sex_male = 1      # Male=1 Female=0
smoker_yes = 0    # Smoker=1 Non-smoker=0

new_data = pd.DataFrame([{
    'age': age,
    'bmi': bmi,
    'children': children,
    'sex_male': sex_male,
    'smoker_yes': smoker_yes
}])

new_scaled = scaler.transform(new_data)
prediction = best_model.predict(new_scaled)

print("Predicted Charges:", prediction[0])

Predicted Charges: 6089.597005378206


In [8]:
# Save model and scaler
with open("svr_grid_model.pkl","wb") as f:
    pickle.dump(best_model,f)

with open("scaler.pkl","wb") as f:
    pickle.dump(scaler,f)

print("Model and scaler saved.")

Model and scaler saved.
